# SFVP AI Clone — Notebook 02: Lip Sync Video
**Tool:** MuseTalk v1.5 (Apache 2.0 — commercial use OK)

Takes Shennel's generated voice audio + a base video of her face → produces a lip-synced video that looks real.

**Drive structure needed:**
```
My Drive/SFVP_Clone/
├── base_videos/     ← your recorded face footage (.mp4)
├── outputs/         ← voice audio from Notebook 01 goes here, video outputs go here too
```

**Runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# CELL 1 — Install MuseTalk + dependencies (run once per session, ~5 min)
import os

# Clone MuseTalk
if not os.path.exists('/content/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk -q

%cd /content/MuseTalk

!pip install -r requirements.txt -q
!pip install --upgrade diffusers transformers accelerate -q
!apt-get install -y ffmpeg -q

print('MuseTalk installed.')

In [ ]:
# CELL 2 — Download MuseTalk model weights from HuggingFace
# This runs once and downloads ~2GB of model weights

!pip install huggingface_hub -q
from huggingface_hub import snapshot_download

os.makedirs('/content/MuseTalk/models/musetalk', exist_ok=True)
os.makedirs('/content/MuseTalk/models/dwpose', exist_ok=True)

print('Downloading MuseTalk weights...')
snapshot_download(
    repo_id='TMElyralab/MuseTalk',
    local_dir='/content/MuseTalk/models/musetalk',
    ignore_patterns=['*.md', '*.txt']
)

print('Downloading face landmark weights...')
snapshot_download(
    repo_id='yzd-v/DWPose',
    local_dir='/content/MuseTalk/models/dwpose',
    ignore_patterns=['*.md']
)

print('All weights downloaded.')

In [ ]:
# CELL 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'

# List base videos
base_dir = f'{DRIVE_BASE}/base_videos'
if os.path.exists(base_dir):
    vids = [f for f in os.listdir(base_dir) if f.endswith(('.mp4','.mov','.avi'))]
    print(f'Found {len(vids)} base video(s):')
    for i, v in enumerate(vids):
        print(f'  [{i}] {v}')
else:
    print('ERROR: No base_videos folder. Upload your face recordings first.')

# List available audio (outputs from Notebook 01)
out_dir = f'{DRIVE_BASE}/outputs'
audios = [f for f in os.listdir(out_dir) if f.endswith('.wav')] if os.path.exists(out_dir) else []
print(f'\nFound {len(audios)} audio file(s) in outputs/:')
for a in audios:
    print(f'  {a}')

In [ ]:
# CELL 4 — CONFIGURE THIS CELL

# Base video of Shennel's face (from base_videos/ folder)
BASE_VIDEO_FILENAME = 'shennel_base_casual.mp4'  # <-- CHANGE THIS

# Audio file to lip-sync to (from outputs/ folder — generated in Notebook 01)
AUDIO_FILENAME = 'voice_output_01.wav'  # <-- CHANGE THIS

# Output video filename
OUTPUT_VIDEO_FILENAME = 'lipsync_output_01.mp4'  # <-- CHANGE IF NEEDED

# MuseTalk quality settings
BBOX_SHIFT = 0       # Adjust if mouth looks off: negative moves up, positive moves down
BATCH_SIZE = 8       # Higher = faster but more VRAM. T4 handles 8 fine.

print('Config set.')
print(f'Base video: {BASE_VIDEO_FILENAME}')
print(f'Audio:      {AUDIO_FILENAME}')
print(f'Output:     {OUTPUT_VIDEO_FILENAME}')

In [ ]:
# CELL 5 — Prepare input files
import subprocess
import shutil

base_video_src = f'{DRIVE_BASE}/base_videos/{BASE_VIDEO_FILENAME}'
audio_src = f'{DRIVE_BASE}/outputs/{AUDIO_FILENAME}'

# Copy to local /content for faster processing
local_video = f'/content/base_video.mp4'
local_audio = f'/content/audio.wav'

shutil.copy(base_video_src, local_video)
shutil.copy(audio_src, local_audio)

# Get video info
result = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', local_video],
    capture_output=True, text=True
)
import json
info = json.loads(result.stdout)
video_stream = next(s for s in info['streams'] if s['codec_type'] == 'video')
print(f'Base video: {video_stream["width"]}x{video_stream["height"]} @ {video_stream.get("r_frame_rate","?")} fps')
print(f'Audio: {local_audio}')
print('Files ready.')

In [ ]:
# CELL 6 — Run MuseTalk lip sync
# This is the main generation step — takes 2-8 min depending on video length

%cd /content/MuseTalk

local_output = f'/content/lipsync_raw.mp4'

cmd = [
    'python', '-m', 'scripts.inference',
    '--video_path', local_video,
    '--audio_path', local_audio,
    '--output_vid_name', local_output,
    '--bbox_shift', str(BBOX_SHIFT),
    '--batch_size', str(BATCH_SIZE),
]

print('Running MuseTalk lip sync...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('ERROR:')
    print(result.stderr[-3000:])  # last 3000 chars of error
else:
    print('Lip sync complete!')
    
    # Copy result to Drive
    drive_output = f'{DRIVE_BASE}/outputs/{OUTPUT_VIDEO_FILENAME}'
    shutil.copy(local_output, drive_output)
    print(f'Saved to Drive: {drive_output}')

In [ ]:
# CELL 7 — Preview output in Colab
from IPython.display import HTML
from base64 import b64encode

with open(local_output, 'rb') as f:
    video_data = f.read()

video_b64 = b64encode(video_data).decode()
HTML(f'''
<video width="480" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
<p><b>How does it look?</b><br>
- If lips look off: adjust BBOX_SHIFT in Cell 4 (try -5 to +5)<br>
- If quality looks grainy: run Notebook 04 to enhance it<br>
- If it looks good: send it to Notebook 04 to polish for posting</p>
''')

In [ ]:
# CELL 8 — (OPTIONAL) Test with multiple base videos side by side
# Useful for finding which base video gives the best results

base_videos = [f for f in os.listdir(f'{DRIVE_BASE}/base_videos') if f.endswith('.mp4')]
print(f'Testing {min(3, len(base_videos))} base videos with same audio...\n')

for i, vid_file in enumerate(base_videos[:3]):
    src = f'{DRIVE_BASE}/base_videos/{vid_file}'
    local_v = f'/content/test_base_{i}.mp4'
    local_out = f'/content/test_lipsync_{i}.mp4'
    shutil.copy(src, local_v)
    
    subprocess.run([
        'python', '-m', 'scripts.inference',
        '--video_path', local_v, '--audio_path', local_audio,
        '--output_vid_name', local_out,
        '--bbox_shift', '0', '--batch_size', '8',
    ], capture_output=True)
    
    print(f'--- Base video [{i}]: {vid_file} ---')
    with open(local_out, 'rb') as f:
        vb64 = b64encode(f.read()).decode()
    display(HTML(f'<video width="320" controls><source src="data:video/mp4;base64,{vb64}" type="video/mp4"></video>'))
    print()